# 🏥 Emergency Department Wait Time Prediction
## Milestone 2: Data Collection, Wrangling, and Visualization

**Course:** CSCI-7090-A-O1E — Data Science & Machine Learning  
**Institution:** Georgia Southern University  
**Team 3:**
- Vandan Patel — Team Lead
- Annagrace Howell — Developer
- Yasmin Rocio Orduz Landazabal — Data Analyst

**Instructor:** Ph.D. Vijayalakshmi Ramasamy  
**Date:** March 2026

---

## Overview

This notebook constitutes Milestone 2 of the Team 3 project, building directly on our Milestone 1 systematic literature review of machine learning approaches for emergency department (ED) wait time prediction. In this milestone we:

1. **Acquire and describe** the MIMIC-IV-ED public dataset from PhysioNet
2. **Wrangle and preprocess** the raw data (missing values, encoding, outliers, feature engineering, scaling)
3. **Conduct exploratory data analysis (EDA)** with publication-quality visualizations

The cleaned dataset produced here will feed directly into the predictive modeling work in Milestone 3.

---
## 0. Environment Setup & Library Installation

In [ ]:
# Install any libraries not available by default in Colab
!pip install -q shap missingno

In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno

# ── Machine-learning utilities ────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from scipy import stats

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Global plot style — colorblind-friendly palette ──────────────────────────
CB_PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
              '#56B4E9', '#D55E00', '#F0E442', '#999999']
sns.set_theme(style='whitegrid', palette=CB_PALETTE, font_scale=1.15)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})

print('✅  All libraries imported successfully.')

---
# Part A — Data Acquisition and Description (10 Points)

## A1. Dataset Source

| Field | Detail |
|---|---|
| **Dataset name** | MIMIC-IV-ED (v2.2) |
| **Provider** | PhysioNet / Beth Israel Deaconess Medical Center, Boston, MA |
| **URL** | https://physionet.org/content/mimic-iv-ed/2.2/ |
| **Access date** | March 2026 |
| **License** | PhysioNet Credentialed Health Data License 1.5.0 |
| **Citation** | Johnson A et al. (2023). *MIMIC-IV-ED (v2.2)*. PhysioNet. https://doi.org/10.13026/5ntk-km72 |

### How to download (run once)
After completing PhysioNet credentialing, run the cell below **or** manually upload the files to your Colab environment.

In [ ]:
# ── Option 1: wget (PhysioNet credentialed download) ─────────────────────────
# Replace YOUR_USERNAME and YOUR_PASSWORD with your PhysioNet credentials.
# !wget -r -N -c -np \
#   --user YOUR_USERNAME --password YOUR_PASSWORD \
#   https://physionet.org/files/mimic-iv-ed/2.2/

# ── Option 2: Upload manually to Colab then set path ─────────────────────────
# from google.colab import files
# uploaded = files.upload()

# ── For reproducibility: point to your local paths ───────────────────────────
TRIAGE_PATH   = 'triage.csv.gz'       # adjust if needed
EDSTAYS_PATH  = 'edstays.csv.gz'
VITALS_PATH   = 'vitalsign.csv.gz'
MEDRECON_PATH = 'medrecon.csv.gz'

print('📂  Dataset paths configured. Proceed to A2 to load data.')

## A2. Load Raw Data & Dataset Description

In [ ]:
# ── Load the two primary files ────────────────────────────────────────────────
triage  = pd.read_csv(TRIAGE_PATH,  compression='gzip', low_memory=False)
edstays = pd.read_csv(EDSTAYS_PATH, compression='gzip', low_memory=False)

print(f'triage  shape : {triage.shape}')
print(f'edstays shape : {edstays.shape}')
print()
print('=== triage.head() ===')
display(triage.head(3))
print('=== edstays.head() ===')
display(edstays.head(3))

In [ ]:
# ── Merge on stay_id ─────────────────────────────────────────────────────────
df_raw = pd.merge(edstays, triage, on='stay_id', how='inner')

# ── Parse timestamps ─────────────────────────────────────────────────────────
df_raw['intime']  = pd.to_datetime(df_raw['intime'])
df_raw['outtime'] = pd.to_datetime(df_raw['outtime'])

# ── Compute wait_time_minutes (target variable) ───────────────────────────────
# In MIMIC-IV-ED 'intime' = arrival; triage timestamps are within triage.csv.
# We use (outtime - intime) as a proxy for total ED length-of-stay,
# consistent with Wu et al. (2025) and Xie et al. (2022).
df_raw['ed_los_minutes'] = (df_raw['outtime'] - df_raw['intime']).dt.total_seconds() / 60

print(f'Merged dataset shape : {df_raw.shape}')
print()
print('Target variable summary:')
display(df_raw['ed_los_minutes'].describe())

In [ ]:
# ── Number of records / features / dtypes ────────────────────────────────────
print(f'Records (rows)  : {df_raw.shape[0]:,}')
print(f'Features (cols) : {df_raw.shape[1]}')
print()
print('Data types per feature:')
display(df_raw.dtypes.reset_index().rename(columns={'index': 'Feature', 0: 'dtype'}))
print()
print('Descriptive statistics (numeric):')
display(df_raw.describe())

## A3. Data Dictionary

| # | Feature | Source Table | Data Type | Description | Example Values |
|---|---------|-------------|-----------|-------------|----------------|
| 1 | `subject_id` | edstays | int64 | De-identified patient identifier | 10000032 |
| 2 | `hadm_id` | edstays | float64 | Hospital admission ID (null if not admitted) | 22532162, NaN |
| 3 | `stay_id` | edstays | int64 | Unique ED stay identifier | 30000000 |
| 4 | `intime` | edstays | datetime64 | ED arrival timestamp | 2112-03-09 04:12:00 |
| 5 | `outtime` | edstays | datetime64 | ED departure timestamp | 2112-03-09 11:45:00 |
| 6 | `gender` | edstays | object | Patient biological sex | M, F |
| 7 | `race` | edstays | object | Patient self-reported race/ethnicity | WHITE, BLACK/AFRICAN AMERICAN |
| 8 | `arrival_transport` | edstays | object | Mode of arrival to ED | WALK IN, AMBULANCE, UNKNOWN |
| 9 | `disposition` | edstays | object | Discharge disposition | HOME, ADMITTED, LEFT WITHOUT BEING SEEN |
| 10 | `temperature` | triage | float64 | Body temperature (°F) at triage | 98.2, 101.4 |
| 11 | `heartrate` | triage | float64 | Heart rate (bpm) at triage | 72, 110 |
| 12 | `resprate` | triage | float64 | Respiratory rate (breaths/min) | 16, 24 |
| 13 | `o2sat` | triage | float64 | Oxygen saturation (%) | 97, 88 |
| 14 | `sbp` | triage | float64 | Systolic blood pressure (mmHg) | 120, 180 |
| 15 | `dbp` | triage | float64 | Diastolic blood pressure (mmHg) | 80, 110 |
| 16 | `pain` | triage | float64 | Self-reported pain score (0–10) | 0, 5, 10 |
| 17 | `acuity` | triage | float64 | ESI triage acuity level (1=most acute, 5=least) | 1, 2, 3, 4, 5 |
| 18 | `chiefcomplaint` | triage | object | Free-text chief complaint | chest pain, shortness of breath |
| 19 | **`ed_los_minutes`** | computed | float64 | **TARGET: Total ED length-of-stay (minutes)** | 245.3, 480.0 |

## A4. Relevance Statement

The MIMIC-IV-ED dataset is ideally suited to our research objective of predicting emergency department wait times using machine learning. Our Milestone 1 systematic literature review identified this dataset as the community reproducibility gold standard, appearing in three of twelve reviewed studies (Xie et al., 2022; Lett et al., 2025; Sun et al., 2024). The dataset contains over 400,000 real ED visits from Beth Israel Deaconess Medical Center (2011–2019) and includes the precise triage variables — ESI acuity level, vital signs, patient demographics, arrival transport mode, and disposition — that the literature consistently identifies as the strongest predictors of ED wait time (Wang et al., 2025; Wu et al., 2025). Critically, unlike the nine proprietary single-center datasets used across the reviewed studies, MIMIC-IV-ED is publicly available via PhysioNet, enabling full reproducibility of our pipeline by other researchers. For Milestone 3, we will train XGBoost, Random Forest, and Gradient Boosting regression models on this cleaned dataset to predict continuous ED length-of-stay in minutes — extending beyond the binary classification approach used by ten of twelve reviewed papers — and apply SHAP-based interpretability and ESI-level subgroup fairness analysis to address the key gaps identified in the literature.

---
# Part B — Data Wrangling and Preprocessing (20 Points)

In [ ]:
# Work on a copy so raw data is always preserved
df = df_raw.copy()

# Keep only clinically relevant columns for this milestone
KEEP_COLS = [
    'subject_id', 'stay_id', 'hadm_id',
    'intime', 'outtime',
    'gender', 'race', 'arrival_transport', 'disposition',
    'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp',
    'pain', 'acuity', 'chiefcomplaint',
    'ed_los_minutes'
]
# Filter to columns that exist (some MIMIC-IV-ED versions vary)
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]
df = df[KEEP_COLS]

print(f'Working dataset shape: {df.shape}')
display(df.head(5))

## B1. Missing Value Analysis

In [ ]:
# ── Quantify missingness ──────────────────────────────────────────────────────
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
}).sort_values('Missing %', ascending=False)

missing = missing[missing['Missing Count'] > 0]
print('Features with missing values:')
display(missing)

# ── Missingness heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
msno.bar(df, ax=ax, color=CB_PALETTE[0], fontsize=11)
ax.set_title('Figure 1 — Completeness Bar Chart per Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_completeness.png', dpi=120)
plt.show()
print('\nFigure 1 Interpretation: Bars shorter than 100% indicate missing data. '
      'Vital signs (temperature, pain, o2sat) show the highest missingness rates, '
      'which is expected in emergency triage settings where not all vitals are measured '
      'for every patient. Columns with >50% missing will be dropped; others will be imputed.')

In [ ]:
# ── Step 1: Drop columns with >50% missing ────────────────────────────────────
threshold = 0.50
cols_to_drop = missing[missing['Missing %'] > threshold * 100].index.tolist()
if cols_to_drop:
    print(f'Dropping high-missingness columns (>{threshold*100:.0f}%): {cols_to_drop}')
    df.drop(columns=cols_to_drop, inplace=True)
else:
    print('No columns exceed the 50% missing threshold.')

# ── Step 2: Drop rows missing the TARGET variable ─────────────────────────────
before = len(df)
df.dropna(subset=['ed_los_minutes'], inplace=True)
print(f'Dropped {before - len(df):,} rows with missing target (ed_los_minutes).')

# ── Step 3: Numeric features — median imputation ─────────────────────────────
# Justification: vital-sign distributions are right-skewed; median is robust to outliers.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols
                if c not in ('subject_id', 'stay_id', 'hadm_id', 'ed_los_minutes')]

num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# ── Step 4: Categorical features — mode (most-frequent) imputation ────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'chiefcomplaint']
for col in cat_cols:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f'  {col}: mode imputation → "{mode_val}"')

# chiefcomplaint: fill with 'UNKNOWN' (text feature; mode would be misleading)
if 'chiefcomplaint' in df.columns:
    df['chiefcomplaint'].fillna('UNKNOWN', inplace=True)

print(f'\nRemaining missing values: {df.isnull().sum().sum()}')
print(f'Dataset shape after imputation: {df.shape}')

## B2. Data Type Conversion

In [ ]:
# ── Timestamps already parsed in A2 ──────────────────────────────────────────
print('Timestamp columns dtype:', df[['intime', 'outtime']].dtypes.to_dict())

# ── acuity → int (1–5 ordinal ESI level) ─────────────────────────────────────
df['acuity'] = df['acuity'].astype(int)

# ── pain → float (already numeric, but force) ────────────────────────────────
df['pain'] = pd.to_numeric(df['pain'], errors='coerce')
df['pain'].fillna(df['pain'].median(), inplace=True)

# ── hadm_id: presence flag (was patient admitted?) ───────────────────────────
df['was_admitted'] = df['hadm_id'].notna().astype(int)

# ── Categorical encoding — Label encoding for high-cardinality columns ────────
# We use label encoding here for EDA; one-hot encoding will be done before modeling.
le = LabelEncoder()

for col in ['gender', 'race', 'arrival_transport', 'disposition']:
    if col in df.columns:
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        print(f'Label-encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print('\nUpdated dtypes:')
display(df.dtypes)

## B3. Outlier Detection and Treatment

In [ ]:
# Columns to inspect for outliers
OUTLIER_COLS = ['ed_los_minutes', 'heartrate', 'sbp', 'dbp',
                'temperature', 'o2sat', 'resprate', 'pain']
OUTLIER_COLS = [c for c in OUTLIER_COLS if c in df.columns]

# ── Method 1: IQR Method ─────────────────────────────────────────────────────
iqr_summary = {}
for col in OUTLIER_COLS:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    iqr_summary[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                         'Lower fence': lo, 'Upper fence': hi,
                         'Outlier count': n_out,
                         'Outlier %': round(n_out / len(df) * 100, 2)}

iqr_df = pd.DataFrame(iqr_summary).T
print('=== IQR Outlier Summary ===')
display(iqr_df)

# ── Method 2: Z-score Method ─────────────────────────────────────────────────
zscore_summary = {}
for col in OUTLIER_COLS:
    z = np.abs(stats.zscore(df[col].dropna()))
    n_out = (z > 3).sum()
    zscore_summary[col] = {'|Z|>3 count': n_out,
                            '|Z|>3 %': round(n_out / len(df) * 100, 2)}

zscore_df = pd.DataFrame(zscore_summary).T
print('\n=== Z-score Outlier Summary (|Z| > 3) ===')
display(zscore_df)

In [ ]:
# ── Box plots for key vital signs (Figure 2) ──────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(OUTLIER_COLS):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=CB_PALETTE[i % len(CB_PALETTE)], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xticks([])

fig.suptitle('Figure 2 — Box Plots for Outlier Detection (Key Features)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig2_boxplots_outliers.png', dpi=120)
plt.show()
print('\nFigure 2 Interpretation: Box plots reveal physiologically implausible '
      'values in several vital signs. Heart rate values above 300 bpm and '
      'systolic BP above 400 mmHg are data entry errors, not real patient values. '
      'For ed_los_minutes, extreme values (>72 hours) likely represent prolonged '
      'boarding episodes rather than typical ED visits.')

In [ ]:
# ── Outlier Treatment: Winsorize (cap) at physiologically plausible limits ────
# Justification: Capping preserves row count while bounding extreme values.
# Clinical reference ranges used as upper/lower bounds.

CLINICAL_BOUNDS = {
    'heartrate':     (20, 300),
    'sbp':           (50, 300),
    'dbp':           (20, 200),
    'temperature':   (85, 110),
    'o2sat':         (50, 100),
    'resprate':      (4, 60),
    'pain':          (0, 10),
    'ed_los_minutes': (5, 60*24*3),  # 5 min – 3 days
}

for col, (lo, hi) in CLINICAL_BOUNDS.items():
    if col in df.columns:
        before = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lower=lo, upper=hi)
        print(f'  {col}: capped {before:,} values to [{lo}, {hi}]')

print(f'\nDataset shape after outlier treatment: {df.shape}')

## B4. Feature Engineering

In [ ]:
# ── Feature 1: Arrival Hour (temporal pattern feature) ───────────────────────
# Rationale: ED crowding peaks at specific hours (e.g., evenings). Arrival hour
# was identified as the top predictor in Wang et al. (2025) and Wang, Sambamoorthi
# et al. (2025). Extracting it from timestamp is essential.
df['arrival_hour'] = df['intime'].dt.hour

# ── Feature 2: Arrival Day of Week ───────────────────────────────────────────
# Rationale: ED volumes differ significantly by weekday vs weekend (Ricciardi et al., 2024).
df['arrival_dow'] = df['intime'].dt.dayofweek  # 0=Monday … 6=Sunday
df['is_weekend']  = (df['arrival_dow'] >= 5).astype(int)

# ── Feature 3: Shock Index (physiological interaction) ───────────────────────
# Rationale: Shock Index = Heart Rate / Systolic BP is a validated clinical
# indicator of hemodynamic instability. Values >1 suggest shock. This interaction
# feature captures patient severity beyond individual vitals.
df['shock_index'] = df['heartrate'] / df['sbp'].replace(0, np.nan)
df['shock_index'].fillna(df['shock_index'].median(), inplace=True)

# ── Feature 4: Pulse Pressure (cardiovascular feature) ───────────────────────
# Rationale: Pulse Pressure = SBP – DBP. Abnormal pulse pressure (too narrow or
# too wide) indicates cardiac pathology and may correlate with resource-intensive
# workups that prolong LOS.
df['pulse_pressure'] = df['sbp'] - df['dbp']

# ── Feature 5: Arrival Shift (categorical time-of-day) ────────────────────────
# Rationale: Hospital operations differ by shift (day/evening/night).
def assign_shift(hour):
    if 7 <= hour < 15:
        return 'Day'
    elif 15 <= hour < 23:
        return 'Evening'
    else:
        return 'Night'

df['arrival_shift'] = df['arrival_hour'].apply(assign_shift)
df['arrival_shift_encoded'] = LabelEncoder().fit_transform(df['arrival_shift'])

# ── Summary of new features ───────────────────────────────────────────────────
new_features = ['arrival_hour', 'arrival_dow', 'is_weekend',
                'shock_index', 'pulse_pressure',
                'arrival_shift', 'arrival_shift_encoded', 'was_admitted']
print('New engineered features summary:')
display(df[new_features].describe(include='all'))
print(f'\nDataset shape after feature engineering: {df.shape}')

## B5. Data Normalization / Scaling

In [ ]:
# ── Numeric features selected for modeling ────────────────────────────────────
MODEL_FEATURES = [
    'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp',
    'pain', 'acuity', 'shock_index', 'pulse_pressure',
    'arrival_hour', 'arrival_dow', 'is_weekend', 'was_admitted',
    'gender_encoded', 'race_encoded', 'arrival_transport_encoded',
    'arrival_shift_encoded'
]
MODEL_FEATURES = [c for c in MODEL_FEATURES if c in df.columns]

# ── Choice: StandardScaler (Z-score normalization) ────────────────────────────
# Justification: Tree-based models (XGBoost, RF) are scale-invariant, but
# StandardScaler ensures fair comparison with distance-based or linear models
# that may be added as baselines in Milestone 3. StandardScaler is preferred
# over MinMaxScaler because our data contains outliers even after capping,
# and z-score is less sensitive to residual extreme values.

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[MODEL_FEATURES] = scaler.fit_transform(df[MODEL_FEATURES])

print('StandardScaler applied. Sample of scaled values:')
display(df_scaled[MODEL_FEATURES].describe().round(3))
print('\nNote: Mean ≈ 0, Std ≈ 1 confirms successful scaling.')

## B6. Final Dataset Summary — Before vs. After Wrangling

In [ ]:
summary_data = {
    'Metric': [
        'Number of rows',
        'Number of columns',
        'Missing value cells',
        'Missing value %',
        'Numeric features',
        'Categorical features',
        'Target (ed_los_minutes) mean (min)',
        'Target std (min)',
        'Target min (min)',
        'Target max (min)',
    ],
    'Before Wrangling': [
        f'{df_raw.shape[0]:,}',
        f'{df_raw.shape[1]}',
        f'{df_raw.isnull().sum().sum():,}',
        f'{df_raw.isnull().mean().mean()*100:.1f}%',
        f'{len(df_raw.select_dtypes(include=np.number).columns)}',
        f'{len(df_raw.select_dtypes(include="object").columns)}',
        f'{df_raw["ed_los_minutes"].mean():.1f}',
        f'{df_raw["ed_los_minutes"].std():.1f}',
        f'{df_raw["ed_los_minutes"].min():.1f}',
        f'{df_raw["ed_los_minutes"].max():.1f}',
    ],
    'After Wrangling': [
        f'{df.shape[0]:,}',
        f'{df.shape[1]}',
        f'{df.isnull().sum().sum():,}',
        f'{df.isnull().mean().mean()*100:.1f}%',
        f'{len(df.select_dtypes(include=np.number).columns)}',
        f'{len(df.select_dtypes(include="object").columns)}',
        f'{df["ed_los_minutes"].mean():.1f}',
        f'{df["ed_los_minutes"].std():.1f}',
        f'{df["ed_los_minutes"].min():.1f}',
        f'{df["ed_los_minutes"].max():.1f}',
    ]
}

summary_df = pd.DataFrame(summary_data)
print('=== Dataset Before vs. After Wrangling ===')
display(summary_df)

# Export the cleaned dataset
df.to_csv('team3_ed_waittime_cleaned.csv', index=False)
print('\n✅  Cleaned dataset saved to: team3_ed_waittime_cleaned.csv')

---
# Part C — Exploratory Data Analysis and Visualization (15 Points)

## C1. Distribution Plots — Key Numerical Features (Figures 3–5)

In [ ]:
# ── Figure 3: Distribution of ED Length-of-Stay (TARGET) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
axes[0].hist(df['ed_los_minutes'], bins=80, color=CB_PALETTE[0],
             edgecolor='white', alpha=0.85, density=True, label='Histogram')
df['ed_los_minutes'].plot.kde(ax=axes[0], color='black', lw=2, label='KDE')
axes[0].axvline(df['ed_los_minutes'].median(), color=CB_PALETTE[1],
                linestyle='--', lw=2, label=f'Median = {df["ed_los_minutes"].median():.0f} min')
axes[0].set_xlabel('ED Length-of-Stay (minutes)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Distribution of ED LOS (minutes)', fontsize=12)
axes[0].legend()

# Log-transformed (right-skewed distributions are easier to read on log scale)
los_log = np.log1p(df['ed_los_minutes'])
axes[1].hist(los_log, bins=60, color=CB_PALETTE[2], edgecolor='white', alpha=0.85, density=True)
los_log.plot.kde(ax=axes[1], color='black', lw=2)
axes[1].set_xlabel('log(ED LOS + 1)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Log-Transformed Distribution of ED LOS', fontsize=12)

fig.suptitle('Figure 3 — ED Length-of-Stay Distribution (Target Variable)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_los_distribution.png', dpi=120)
plt.show()
print('Figure 3 Interpretation: ED LOS is strongly right-skewed (mean > median), '
      'with most visits concentrated below 600 minutes (10 hours). '
      'A long tail of extreme values represents complex or boarding cases. '
      'The log-transformed version approaches normality, suggesting a '
      'log-transformation may benefit regression modeling in Milestone 3.')

In [ ]:
# ── Figure 4: Distribution of Triage Vital Signs ─────────────────────────────
VITALS = ['heartrate', 'sbp', 'o2sat', 'resprate']
VITALS = [v for v in VITALS if v in df.columns]

fig, axes = plt.subplots(1, len(VITALS), figsize=(16, 4))
for i, col in enumerate(VITALS):
    axes[i].hist(df[col].dropna(), bins=50,
                 color=CB_PALETTE[i], edgecolor='white', alpha=0.85, density=True)
    df[col].dropna().plot.kde(ax=axes[i], color='black', lw=1.5)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('Value', fontsize=10)
    axes[i].set_ylabel('Density' if i == 0 else '', fontsize=10)

fig.suptitle('Figure 4 — Distribution of Key Triage Vital Signs',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_vitals_distributions.png', dpi=120)
plt.show()
print('Figure 4 Interpretation: Heart rate follows a roughly normal distribution '
      'centered near 80 bpm. Systolic blood pressure peaks around 120 mmHg with '
      'a tail indicating hypertensive patients. Oxygen saturation is heavily '
      'left-skewed: the vast majority of patients present with normal saturation '
      '(95–100%), but a clinically important minority show dangerous desaturation. '
      'Respiratory rate is near-normally distributed around 18 breaths/min.')

In [ ]:
# ── Figure 5: Distribution of Engineered Features ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shock Index
axes[0].hist(df['shock_index'].clip(0, 3), bins=60,
             color=CB_PALETTE[4], edgecolor='white', alpha=0.85, density=True)
df['shock_index'].clip(0, 3).plot.kde(ax=axes[0], color='black', lw=1.5)
axes[0].axvline(1.0, color=CB_PALETTE[5], linestyle='--', lw=2, label='Shock threshold (SI=1.0)')
axes[0].set_title('Shock Index Distribution', fontsize=12)
axes[0].set_xlabel('Shock Index (HR / SBP)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].legend()

# Pulse Pressure
axes[1].hist(df['pulse_pressure'], bins=60,
             color=CB_PALETTE[3], edgecolor='white', alpha=0.85, density=True)
df['pulse_pressure'].plot.kde(ax=axes[1], color='black', lw=1.5)
axes[1].axvline(40, color='green', linestyle='--', lw=2, label='Normal PP = 40 mmHg')
axes[1].set_title('Pulse Pressure Distribution', fontsize=12)
axes[1].set_xlabel('Pulse Pressure (SBP − DBP, mmHg)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].legend()

fig.suptitle('Figure 5 — Distribution of Engineered Clinical Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_engineered_dist.png', dpi=120)
plt.show()
print('Figure 5 Interpretation: The Shock Index distribution is right-skewed and '
      'centered below 1.0, indicating most patients are hemodynamically stable at '
      'triage. However, a visible proportion exceed SI=1.0, signalling potential '
      'shock — these patients typically require more resources and longer stays. '
      'Pulse Pressure centers around 40 mmHg (normal), with tails representing '
      'patients with cardiovascular pathologies.')

## C2. Correlation Heatmap (Figure 6)

In [ ]:
# ── Select numeric modeling columns ───────────────────────────────────────────
CORR_COLS = [
    'ed_los_minutes', 'acuity', 'heartrate', 'sbp', 'dbp',
    'temperature', 'o2sat', 'resprate', 'pain',
    'shock_index', 'pulse_pressure',
    'arrival_hour', 'is_weekend', 'was_admitted'
]
CORR_COLS = [c for c in CORR_COLS if c in df.columns]

corr_matrix = df[CORR_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Figure 6 — Pearson Correlation Heatmap (Numeric Features)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig6_correlation_heatmap.png', dpi=120)
plt.show()

# Top correlations with target
target_corr = corr_matrix['ed_los_minutes'].drop('ed_los_minutes').sort_values(key=abs, ascending=False)
print('\nTop correlations with ed_los_minutes:')
display(target_corr.to_frame())
print('\nFigure 6 Interpretation: The strongest positive correlation with ED LOS is '
      'was_admitted (hospitalized patients stay far longer). Acuity (ESI level) '
      'correlates inversely with LOS — lower ESI numbers mean higher acuity and '
      'paradoxically longer stays due to intensive workups. SBP and DBP are '
      'strongly correlated with each other (multicollinearity), which motivates '
      'the pulse_pressure interaction feature. Pain score has a modest positive '
      'correlation with LOS, reflecting that higher pain is associated with '
      'more complex presentations.')

## C3. Categorical Feature Analysis (Figures 7 & 8)

In [ ]:
# ── Figure 7: ESI Acuity Level Distribution + Median LOS by ESI ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
acuity_counts = df['acuity'].value_counts().sort_index()
axes[0].bar(acuity_counts.index.astype(str), acuity_counts.values,
            color=CB_PALETTE[:5], edgecolor='white', alpha=0.9)
axes[0].set_xlabel('ESI Acuity Level (1=Most Acute, 5=Least Acute)', fontsize=11)
axes[0].set_ylabel('Number of ED Visits', fontsize=11)
axes[0].set_title('ED Visits by ESI Triage Level', fontsize=12)
for i, (x, y) in enumerate(zip(acuity_counts.index, acuity_counts.values)):
    axes[0].text(i, y + acuity_counts.max() * 0.01, f'{y:,}',
                 ha='center', fontsize=9)

# Median LOS by acuity
los_by_acuity = df.groupby('acuity')['ed_los_minutes'].median()
axes[1].bar(los_by_acuity.index.astype(str), los_by_acuity.values,
            color=CB_PALETTE[:5], edgecolor='white', alpha=0.9)
axes[1].set_xlabel('ESI Acuity Level', fontsize=11)
axes[1].set_ylabel('Median ED LOS (minutes)', fontsize=11)
axes[1].set_title('Median ED LOS by ESI Triage Level', fontsize=12)
for i, (x, y) in enumerate(zip(los_by_acuity.index, los_by_acuity.values)):
    axes[1].text(i, y + 5, f'{y:.0f} min', ha='center', fontsize=9)

fig.suptitle('Figure 7 — ESI Triage Level: Visit Counts and Median LOS',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_acuity_analysis.png', dpi=120)
plt.show()
print('Figure 7 Interpretation: ESI level 3 (urgent, not immediately life-threatening) '
      'accounts for the majority of ED visits, consistent with the literature (Wang et al., '
      '2025 focused exclusively on ESI-3). Counterintuitively, ESI level 1 (most critical) '
      'may show high LOS due to ICU transfers and complex resuscitations, while '
      'ESI level 5 (non-urgent) patients have shorter stays. This supports our '
      'Milestone 3 plan to train separate models per ESI level.')

In [ ]:
# ── Figure 8: Arrival Transport Mode & Disposition ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Arrival transport
if 'arrival_transport' in df.columns:
    transport_counts = df['arrival_transport'].value_counts().head(6)
    wedges, texts, autotexts = axes[0].pie(
        transport_counts.values,
        labels=transport_counts.index,
        autopct='%1.1f%%',
        colors=CB_PALETTE[:len(transport_counts)],
        startangle=140
    )
    axes[0].set_title('Arrival Transport Mode', fontsize=12)

# Disposition
if 'disposition' in df.columns:
    disp_counts = df['disposition'].value_counts().head(7)
    axes[1].barh(disp_counts.index[::-1], disp_counts.values[::-1],
                 color=CB_PALETTE[:len(disp_counts)], alpha=0.9, edgecolor='white')
    axes[1].set_xlabel('Number of Visits', fontsize=11)
    axes[1].set_title('ED Disposition Distribution', fontsize=12)

fig.suptitle('Figure 8 — Categorical Features: Arrival Mode & Disposition',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_categorical_analysis.png', dpi=120)
plt.show()
print('Figure 8 Interpretation: The majority of patients arrive by walk-in, '
      'reflecting the high proportion of lower-acuity visits in the dataset. '
      'Ambulance arrivals represent a smaller but clinically important subset '
      'with typically higher severity and longer stays (confirmed by Wang, '
      'Sambamoorthi et al., 2025 who found ambulance mode was a top SHAP predictor). '
      'The disposition chart shows home discharge dominates, with admitted patients '
      'representing the subgroup with longest LOS.')

## C4. Scatter Plot — Features vs. Target Variable (Figure 9)

In [ ]:
# ── Figure 9: Scatter plots of key numeric features vs. ED LOS ───────────────
SCATTER_FEATURES = ['acuity', 'heartrate', 'sbp', 'pain', 'shock_index']
SCATTER_FEATURES = [f for f in SCATTER_FEATURES if f in df.columns]

fig, axes = plt.subplots(1, len(SCATTER_FEATURES), figsize=(18, 4))

for i, feat in enumerate(SCATTER_FEATURES):
    sample = df.sample(min(5000, len(df)), random_state=42)  # sample for speed
    axes[i].scatter(sample[feat], sample['ed_los_minutes'],
                    alpha=0.15, color=CB_PALETTE[i], s=10)
    # Trend line
    z = np.polyfit(sample[feat].dropna(),
                   sample.loc[sample[feat].notna(), 'ed_los_minutes'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[feat].min(), sample[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), 'k--', lw=1.5, label='Trend')
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('ED LOS (min)' if i == 0 else '', fontsize=10)
    axes[i].set_title(f'{feat} vs LOS', fontsize=10)
    r = df[[feat, 'ed_los_minutes']].corr().iloc[0, 1]
    axes[i].text(0.05, 0.93, f'r = {r:.2f}', transform=axes[i].transAxes,
                 fontsize=9, color='black')

fig.suptitle('Figure 9 — Scatter Plots: Key Features vs. ED LOS (Target)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_scatter_vs_los.png', dpi=120)
plt.show()
print('Figure 9 Interpretation: Acuity shows a modest negative linear relationship '
      'with LOS (higher acuity numbers = less severe = shorter stay). Heart rate '
      'and SBP show relatively weak linear correlations with LOS individually, '
      'confirming that clinical complexity requires the interaction features '
      '(shock index, pulse pressure) engineered in Part B. Pain score shows a '
      'slight positive trend. The wide scatter in all plots indicates that no '
      'single feature explains LOS in isolation — ensemble models capturing '
      'feature interactions will be essential in Milestone 3.')

## C5. Time Series / Temporal Trend Analysis (Figures 10 & 11)

In [ ]:
# ── Figure 10: Average ED LOS by Arrival Hour ─────────────────────────────────
hourly_stats = df.groupby('arrival_hour')['ed_los_minutes'].agg(['mean', 'median', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean LOS by hour
axes[0].bar(hourly_stats.index, hourly_stats['mean'],
            color=CB_PALETTE[0], alpha=0.8, edgecolor='white')
axes[0].plot(hourly_stats.index, hourly_stats['median'],
             color=CB_PALETTE[1], marker='o', ms=5, lw=2, label='Median LOS')
axes[0].set_xlabel('Arrival Hour (0–23)', fontsize=11)
axes[0].set_ylabel('ED LOS (minutes)', fontsize=11)
axes[0].set_title('Average & Median ED LOS by Arrival Hour', fontsize=12)
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

# Visit volume by hour
axes[1].bar(hourly_stats.index, hourly_stats['count'],
            color=CB_PALETTE[2], alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Arrival Hour (0–23)', fontsize=11)
axes[1].set_ylabel('Number of Visits', fontsize=11)
axes[1].set_title('ED Visit Volume by Arrival Hour', fontsize=12)
axes[1].set_xticks(range(0, 24, 2))

fig.suptitle('Figure 10 — Temporal Pattern: ED LOS and Volume by Arrival Hour',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig10_hourly_pattern.png', dpi=120)
plt.show()
print('Figure 10 Interpretation: Visit volume peaks during daytime hours (10am–8pm), '
      'consistent with population activity patterns. LOS tends to be longer for '
      'patients arriving in early morning hours (2–6am), likely because these '
      'patients are more seriously ill (non-urgent patients typically do not seek '
      'ED care at 3am). Evening arrivals (5–9pm) coincide with the highest volumes, '
      'which may increase wait times due to crowding. This validates arrival_hour '
      'and arrival_shift as important predictors for Milestone 3.')

In [ ]:
# ── Figure 11: Monthly Trend in Median ED LOS (2011–2019) ─────────────────────
df['year_month'] = df['intime'].dt.to_period('M')
monthly_stats = df.groupby('year_month')['ed_los_minutes'].agg(['median', 'count'])
monthly_stats.index = monthly_stats.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly_stats.index, monthly_stats['median'],
             color=CB_PALETTE[0], lw=2)
axes[0].fill_between(monthly_stats.index, monthly_stats['median'],
                     alpha=0.2, color=CB_PALETTE[0])
axes[0].set_ylabel('Median ED LOS (min)', fontsize=11)
axes[0].set_title('Monthly Median ED Length-of-Stay (2011–2019)', fontsize=12)

axes[1].bar(monthly_stats.index, monthly_stats['count'],
            color=CB_PALETTE[2], alpha=0.7, width=20)
axes[1].set_ylabel('Monthly Visit Count', fontsize=11)
axes[1].set_title('Monthly ED Visit Volume', fontsize=12)
axes[1].set_xlabel('Date', fontsize=11)

fig.suptitle('Figure 11 — Time-Series Trend: Median LOS and Volume (2011–2019)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_monthly_trend.png', dpi=120)
plt.show()
print('Figure 11 Interpretation: The time-series reveals a gradual upward trend '
      'in median ED LOS over the 2011–2019 period, consistent with the national '
      'trend of increasing ED overcrowding. Visit volume also shows gradual growth '
      'over time. Seasonal patterns (higher volume in winter months) are visible. '
      'This trend motivates careful temporal train-test splitting in Milestone 3 '
      'to prevent data leakage from future to past.')

## C6. Fairness Preview — ED LOS by Demographic Subgroups (Figure 12)

In [ ]:
# ── Figure 12: LOS Distribution by Gender and by Arrival Transport ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LOS by gender
if 'gender' in df.columns:
    gender_groups = [df[df['gender'] == g]['ed_los_minutes'].dropna()
                     for g in df['gender'].unique() if g in ('M', 'F')]
    gender_labels = [g for g in df['gender'].unique() if g in ('M', 'F')]
    axes[0].violinplot(gender_groups, positions=range(len(gender_labels)),
                       showmedians=True)
    axes[0].set_xticks(range(len(gender_labels)))
    axes[0].set_xticklabels(['Female' if g == 'F' else 'Male'
                              for g in gender_labels])
    axes[0].set_ylabel('ED LOS (minutes)', fontsize=11)
    axes[0].set_title('ED LOS Distribution by Gender', fontsize=12)

# LOS by arrival transport
if 'arrival_transport' in df.columns:
    top_transport = df['arrival_transport'].value_counts().head(4).index.tolist()
    transport_los = df[df['arrival_transport'].isin(top_transport)]
    transport_los.boxplot(column='ed_los_minutes', by='arrival_transport',
                          ax=axes[1], notch=False,
                          boxprops=dict(color=CB_PALETTE[0]),
                          medianprops=dict(color='red', lw=2))
    axes[1].set_title('ED LOS by Arrival Transport Mode', fontsize=12)
    axes[1].set_xlabel('Arrival Transport', fontsize=11)
    axes[1].set_ylabel('ED LOS (minutes)', fontsize=11)
    plt.suptitle('')  # suppress default boxplot title

fig.suptitle('Figure 12 — Fairness Preview: LOS by Gender and Arrival Mode',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig12_fairness_preview.png', dpi=120)
plt.show()
print('Figure 12 Interpretation: The violin plot reveals whether gender differences '
      'exist in ED LOS distributions — if present, this would motivate the formal '
      'fairness analysis planned for Milestone 3 (building on Lett et al., 2025 '
      'and Wang et al., 2025). Arrival transport reveals that ambulance patients '
      'have a wider and generally longer LOS distribution than walk-in patients, '
      'confirming this as an important feature and highlighting potential disparities '
      'for patients without transportation access who must walk in.')

---
## Final Export — Cleaned Dataset

In [ ]:
# ── Export cleaned (unscaled) dataset for Milestone 3 modeling ───────────────
export_cols = MODEL_FEATURES + ['ed_los_minutes',
                                 'subject_id', 'stay_id',
                                 'gender', 'race', 'arrival_transport', 'disposition',
                                 'acuity', 'arrival_hour', 'arrival_shift']
export_cols = [c for c in export_cols if c in df.columns]
export_cols = list(dict.fromkeys(export_cols))  # deduplicate preserving order

df_export = df[export_cols].copy()

df_export.to_csv('team3_ed_waittime_cleaned.csv', index=False)

print(f'✅  Final cleaned dataset exported.')
print(f'   Shape : {df_export.shape}')
print(f'   File  : team3_ed_waittime_cleaned.csv')
print()
print('Final column list:')
print(df_export.columns.tolist())
print()
print('Final dataset head:')
display(df_export.head())

---
## References

1. Johnson, A., et al. (2023). *MIMIC-IV-ED (Version 2.2)*. PhysioNet. https://doi.org/10.13026/5ntk-km72
2. Xie, F., Zhou, J., Lee, J. W., et al. (2022). Benchmarking emergency department prediction models with machine learning and public electronic health records. *Scientific Data, 9*, 658. https://doi.org/10.1038/s41597-022-01782-9
3. Wang, et al. (2025). Evaluating fairness of machine learning prediction of prolonged wait times in emergency department with interpretable eXtreme gradient boosting. *PLOS Digital Health, 4*(3). https://doi.org/10.1371/journal.pdig.0000751
4. Wang, H., Sambamoorthi, N., Sandlin, D., & Sambamoorthi, U. (2025). Interpretable machine learning models for prolonged emergency department wait time prediction. *BMC Health Services Research, 25*, 403. https://doi.org/10.1186/s12913-025-12535-w
5. Wu, W., Li, M., Jiang, H., et al. (2025). Development of an emergency department length-of-stay prediction model based on machine learning. *World Journal of Emergency Medicine, 16*(3), 220–224. https://doi.org/10.5847/wjem.j.1920-8642.2025.048
6. Lett, E., Shahbandegan, S., Barak-Corren, Y., Fine, A. M., & La Cava, W. G. (2025). Intersectional and marginal debiasing in prediction models for emergency admissions. *JAMA Network Open, 8*(5), e2512947. https://doi.org/10.1001/jamanetworkopen.2025.12947
7. Ricciardi, C., et al. (2024). Evaluation of different machine learning algorithms for predicting the length of stay in the emergency departments. *Frontiers in Digital Health, 5*, 1323849. https://doi.org/10.3389/fdgth.2023.1323849
8. Sun, L., Agarwal, A., Kornblith, A., Yu, B., & Xiong, C. (2024). ED-Copilot: Reduce emergency department wait time with language model diagnostic assistance. *ICML 2024*, PMLR 235:46942–46956.
9. McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
10. Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*.
11. Pandas documentation: https://pandas.pydata.org/docs/
12. Seaborn documentation: https://seaborn.pydata.org/